<a href="https://colab.research.google.com/github/bduff4/GenAI/blob/main/HW4/HW4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

* Brennan Duff
* Generative AI
* 3/17/2026
* Assignment 4, Advanced RAG Strategies and Retrieval Evaluation, the purpose of this assignment is to observe and report on how RAG retrieval strageties, namely Naive Similarity, MMR (Maximum Marginal Relevance), and Hybrid Search, affect the accuracy and context awarness of a RAG system.

In [1]:
!pip install -qU langchain-core langchain-community langchain-google-genai langchain-classic
!pip install -qU faiss-cpu pypdf rank_bm25 flashrank

# Standard imports for environment management
import os
from google.colab import userdata

# API Key from Colab Secrets
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.5/503.5 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 359.3/359.3 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 141.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.7/345.7 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 613.9/613.9 kB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.4/13

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os

# Updated list to match your specific filenames
file_names = [
    "F14 history and introduction DCS.pdf",
    "F15 history and introduction DCS.pdf",
    "F15E history and introduction DCS.pdf",
    "F16 history and introduction DCS.pdf",
    "FA18 history and introduction DCS.pdf"
]

all_docs = []

# Loop through each file, load its content, and split it into chunks
for file in file_names:
    if os.path.exists(file):
        print(f"Processing {file}...")
        loader = PyPDFLoader(file)

        # Recursive splitter attempts to keep paragraphs and sentences together
        # chunk_size is total characters; overlap keeps context across boundaries
        splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=80)
        chunks = loader.load_and_split(splitter)

        all_docs.extend(chunks)
    else:
        print(f"Warning: '{file}' not found in the sidebar. Please upload it.")

if all_docs:
    print(f"\nSuccess! Total text chunks created: {len(all_docs)}")

Processing F14 history and introduction DCS.pdf...
Processing F15 history and introduction DCS.pdf...
Processing F15E history and introduction DCS.pdf...
Processing F16 history and introduction DCS.pdf...
Processing FA18 history and introduction DCS.pdf...

Success! Total text chunks created: 85


In [3]:
import time
from tenacity import retry, stop_after_attempt, wait_random_exponential, retry_if_exception_type
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

# Initialize Gemini Embeddings (2026 Stable Model)
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    task_type="retrieval_document"
)



# Define the Retry-Safe Embedding Wrapper
# This function will wait and retry automatically if we get a 429 error.
@retry(
    wait=wait_random_exponential(min=1, max=60),
    stop=stop_after_attempt(5),
    retry=retry_if_exception_type(Exception) # Catch the GoogleGenerativeAIError
)
def embed_with_retry(vector_store, batch):
    if vector_store is None:
        return FAISS.from_documents(batch, embeddings)
    else:
        vector_store.add_documents(batch)
        return vector_store



# Process with Batching and Retries
batch_size = 15 # Smaller batches help prevent hitting the limit too fast
vectorstore = None

print(f"Embedding {len(all_docs)} chunks with rate-limit protection...")
for i in range(0, len(all_docs), batch_size):
    batch = all_docs[i : i + batch_size]
    try:
        vectorstore = embed_with_retry(vectorstore, batch)
        print(f" Processed {i + len(batch)}/{len(all_docs)}...")
    except Exception as e:
        print(f" Failed after retries: {e}")
        break
    time.sleep(3) # A steady 3-second heartbeat to keep the API happy




# Naive similarity retriever
similarity_retriever = vectorstore.as_retriever(
    search_kwargs={"k":5}
)

# MMR retriever
mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k":5,
        "fetch_k":20
    }
)

# Keyword retriever
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 5

# Hybrid retriever
hybrid_retriever = EnsembleRetriever(
    retrievers=[similarity_retriever, bm25_retriever],
    weights=[0.5,0.5]
)

print("Retrievers initialized.")


Embedding 85 chunks with rate-limit protection...
 Processed 15/85...
 Processed 30/85...
 Processed 45/85...
 Processed 60/85...
 Processed 75/85...
 Failed after retries: RetryError[<Future at 0x7989de1a98b0 state=finished raised GoogleGenerativeAIError>]
Retrievers initialized.


In [20]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

# Hub import for modular LangChain
try:
    from langchain_classic import hub
except ImportError:
    import langchainhub as hub

# Gemini 2.5 Flash is the 'sweet spot' for speed and availability.
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0.0
)



# Attach Rate-Limit Protection
llm_with_retry = llm.with_retry(
    stop_after_attempt=5,
    wait_exponential_jitter=True
)



# Pull the standard RAG prompt
prompt = hub.pull("rlm/rag-prompt")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs) # formatting docs into string

# THE LCEL PIPELINE (The "Pipe" Syntax)
def build_rag_chain(retriever):

    rag_chain = (
        RunnableParallel(
            {
                "context": retriever | format_docs, # retrieve & format docs
                "question": RunnablePassthrough() # pass question
            }
        )
        | prompt # prompt template
        | llm # send to llm
        | StrOutputParser() # convert output to string
    )

    return rag_chain



In [24]:
# Queries
queries = {
    "Factual": "What is the official name of the F-16?",

    "Conceptual": "Why did the FX program start?",

    "Technical": "At what altitude is the F-14A more susceptible to compressor stalls?"
}



In [22]:
print("\nTRIAL A — Naive Similarity Retrieval")

rag_chain = build_rag_chain(similarity_retriever) # build rag chain

for qtype, query in queries.items(): # loop thru queries

    print("\n--------------------------------")
    print(qtype, "Question")
    print("--------------------------------")

    answer = rag_chain.invoke(query) # generate answer

    print("Q:", query) # print
    print("A:", answer)



TRIAL A — Naive Similarity Retrieval

--------------------------------
Factual Question
--------------------------------
Q: What is the official name of the F-16?
A: The official name of the F-16 is the F-16 Fighting Falcon. It is also nicknamed "Viper" by its pilots.

--------------------------------
Conceptual Question
--------------------------------
Q: Why did the FX program start?
A: The FX program was initiated by the USAF to pave the way for a new generation of fighter aircraft. Its main goal was to create a new fleet of fighters to replace current-generation aircraft and ensure the USAF remained competitive and maintained air superiority. This was partly in response to new Soviet fighter developments like the MiG-25.

--------------------------------
Technical Question
--------------------------------
Q: At what altitude is the F-14A more susceptible to compressor stalls?
A: The F-14A's TF30 engine was susceptible to compressor stalls above 30,000 ft. This susceptibility was a

In [25]:
print("\nTRIAL B — MMR Retrieval")

rag_chain = build_rag_chain(mmr_retriever) # same as above, but MMR

for qtype, query in queries.items():

    print("\n--------------------------------")
    print(qtype, "Question")
    print("--------------------------------")

    answer = rag_chain.invoke(query)

    print("Q:", query)
    print("A:", answer)



TRIAL B — MMR Retrieval

--------------------------------
Factual Question
--------------------------------
Q: What is the official name of the F-16?
A: The official name of the F-16 is the F-16 Fighting Falcon. It was adopted as the F-16 Fighting Falcon in 1974.

--------------------------------
Conceptual Question
--------------------------------
Q: Why did the FX program start?
A: The FX program was initiated by the USAF to pave the way for a new generation of fighter aircraft. Its main goal was to create a new fleet of fighters to replace current-generation aircraft and ensure the USAF remained competitive and maintained air superiority. This program was spurred by the "Fighter Mafia" and John Boyd's 'energy-maneuverability' theory, which emphasized the importance of lightweight, highly maneuverable aircraft.

--------------------------------
Technical Question
--------------------------------
Q: At what altitude is the F-14A more susceptible to compressor stalls?
A: The F-14A's T

In [23]:
print("\nTRIAL C — Hybrid Retrieval")

rag_chain = build_rag_chain(hybrid_retriever) # same as above, but Hybrid

for qtype, query in queries.items():

    print("\n--------------------------------")
    print(qtype, "Question")
    print("--------------------------------")

    answer = rag_chain.invoke(query)

    print("Q:", query)
    print("A:", answer)



TRIAL C — Hybrid Retrieval

--------------------------------
Factual Question
--------------------------------
Q: What is the official name of the F-16?
A: The official name of the F-16 is the F-16 Fighting Falcon. It is also nicknamed "Viper" by its pilots.

--------------------------------
Conceptual Question
--------------------------------
Q: Why did the FX program start?
A: The FX program was initiated by the USAF to pave the way for a new generation of fighter aircraft. Its main goal was to create a new fleet of fighters to replace current-generation aircraft and ensure the USAF remained competitive and maintained air superiority. This was partly in response to new Soviet fighter developments like the MiG-25 Foxbat.

--------------------------------
Technical Question
--------------------------------
Q: At what altitude is the F-14A more susceptible to compressor stalls?
A: The F-14A's TF30 engine was susceptible to compressor stalls at high angles of attack and during rapid thr

# **Analysis**

Compare Trial A to Trial B. Did the Naive approach return three chunks that all said the same thing? Did MMR provide a more "complete" picture for your conceptual question?
* The individual chunks returned by the naive approach were all different. MMR attempted to provide a more complete picture of the conceptual question, however it ended up hallucinating to compensate. (Note: while the "Fighter Mafia" was directly involved in the FX program, nothing in the documents provided pointed to their involvement.)

In Trial C, did the Hybrid search find information that the Vector-only searches (A and B) missed? Hint. Look for specific dates, names, or technical IDs in your documents.
* The Hybrid Search did not differ wildly from the other retrievers, likely due to the simplicity of the prompts. It gathered different information related to the F-14A, specifically that it had engine problems from the start of adoption. Additionally, it gathered the specific model name of the MiG-25 where the naive and MMR retrievers did not. The MMR retriever omitted the nickname of the F-16, likely due to similarity to the desired information.

If a retriever failed to find the answer, did the LLM admit it didn't know, or did it use its internal knowledge to "guess"? Explain how retrieval quality directly impacts "Groundedness."
* When I was prompting for information about the F/A-18 during testing, I was specific enough in my query to confuse the retrievers. This caused the LLM to admit it did not know the answer, likely as the temperature is set to 0. If the quality of a retrieval is subpar, the model will have to answer based on limited information, or in worse cases, guess, lowering accuracy. As such, the quality of retrieval directly impacts groundedness.

Based on your specific dataset, which retrieval strategy would you put into production and why?
* The retrieval strategy I would use for my dataset is the Hybrid Search. In aviation and flight simulation, there are a lot of nicknames, model numbers, techincal terms, etc. and finding these is a strength of the Hybrid Search.

Trial | Factual Response Quality | Conceptual Response Quality | Technical Response Quality|  
-------------------|------------------|-------------------|------------------|
A | 5 | 4.5 | 5 |
B | 4.5 | 2.5 | 4.5 |
C | 5 | 5 | 4.5 |

